# Comparing This Repo's Three Real Tokenizers

Goal: understand tokenization by actually running it, on three tokenizers that already exist in this repo with real, different design choices behind them — not a toy example built to be tidy.

| Tokenizer | Vocab size | Notable trait |
| --- | --- | --- |
| `custom-gpt-6m` | 4,096 | Smallest vocab, has an explicit `<unk>` fallback |
| `custom-gpt-50m` | 50,257 | Real, unmodified public GPT-2 vocabulary (via `tiktoken`) |
| `custom-gpt-350m` | 32,768 | Splits every digit individually, no `<unk>` — pure byte-level fallback |

Read [`../../docs/llm-engineering/09_tokenization.md`](../../docs/llm-engineering/09_tokenization.md) first if BPE/byte-level/merge-order aren't already solid — this notebook assumes that mechanism, it doesn't re-teach it.

Each exercise below has a **prediction prompt before the code cell** — write down what you expect before running it. The interesting part is where you're wrong, not where you're right.

In [ ]:
## Setup — loading the three real tokenizers (this cell is boilerplate, not the exercise)

from pathlib import Path
import tiktoken
from tokenizers import Tokenizer

REPO = Path("../..").resolve()

tok_6m = Tokenizer.from_file(str(REPO / "from_scratch/custom-gpt-6m/data/tokenizer.json"))
tok_350m = Tokenizer.from_file(str(REPO / "from_scratch/custom-gpt-350m/tokenizer/tokenizer.json"))
tok_50m = tiktoken.get_encoding("gpt2")

print(f"6m   vocab size: {tok_6m.get_vocab_size():,}")
print(f"350m vocab size: {tok_350m.get_vocab_size():,}")
print(f"50m  vocab size: {tok_50m.n_vocab:,}")

In [ ]:
## A shared helper — `tokenizers` and `tiktoken` have different APIs, this evens them out
## (also boilerplate — the point is what it PRINTS, not this function itself)

def show(name, text, ids, pieces):
    print(f"--- {name}: {len(ids)} tokens ---")
    print(pieces)
    print()

def encode_all(text):
    ids_6m = tok_6m.encode(text).ids
    pieces_6m = [tok_6m.id_to_token(i) for i in ids_6m]
    show("6m", text, ids_6m, pieces_6m)

    ids_350m = tok_350m.encode(text).ids
    pieces_350m = [tok_350m.id_to_token(i) for i in ids_350m]
    show("350m", text, ids_350m, pieces_350m)

    ids_50m = tok_50m.encode(text)
    pieces_50m = [tok_50m.decode([i]) for i in ids_50m]
    show("50m (GPT-2)", text, ids_50m, pieces_50m)

# Sanity check on a plain sentence
encode_all("The little rabbit hopped through the forest.")

## Exercise 1 — Compression: does a bigger vocab always win?

**Predict first**: pick a longer paragraph of plain English prose (3-4 sentences). Before running anything, rank the three tokenizers by which you expect to produce the **fewest** tokens for it, and write down *why* — is it purely about vocab size, or something else?

Then: paste your paragraph below, run `encode_all`, and compute **tokens ÷ characters** for each (lower = better compression). Was your ranking right? If not, what does that tell you about what actually drives compression besides vocab size — think about what each vocab was actually *trained on* (350m's `oxide-bpe-32k` was trained on this repo's own prose+chat corpus; 50m's is OpenAI's original GPT-2 training mix; 6m's is TinyStories specifically).

In [ ]:
# TODO: paste your own paragraph here and run it
paragraph = """..."""

encode_all(paragraph)

# TODO: compute tokens/character for each tokenizer and compare to your prediction
# hint: len(paragraph) gives you the character count

## Exercise 2 — Digit-splitting: why 350m deliberately tokenizes numbers differently

`custom-gpt-350m`'s tokenizer pre-tokenizes with `Digits(individual_digits=True)` *before* BPE merging even runs — every digit becomes its own token, on purpose (see that project's `src/gpt/tokenizer.py` docstring for the full arithmetic argument).

**Predict first**: encode `"2024"` and `"99999"` with all three. Which tokenizer(s) will split every digit? Which might instead learn "2024" as a single merged token, or split it inconsistently (e.g. "99" + "999")? What would inconsistent splitting actually cost a model trying to learn arithmetic — think about what "learning to add" requires if the same number can appear as a different number of tokens depending on context.

Run it below and check.

In [ ]:
# TODO: try a few different numbers — does "2024" split the same way "99999" does?
# Also try a number that's already appeared many times in training data vs. an unusual one
# (e.g. "1999" vs "48291") — does frequency change anything for the non-digit-split tokenizers?

encode_all("2024")
encode_all("99999")

### A real nuance worth not missing

Running this, you'll find **both** `custom-gpt-6m` and `custom-gpt-350m` split every digit individually, while `custom-gpt-50m` (real GPT-2) doesn't — `"2024"` becomes `["20","24"]` but `"99999"` becomes `["99","999"]`, the exact inconsistent-segmentation problem 350m's tokenizer was built to avoid.

But 6m and 350m arrive at the same *surface behavior* for completely different *reasons* — worth distinguishing, not treating as "two examples of the same design choice":

- **350m**: deliberate — `Digits(individual_digits=True)` runs before BPE even sees the text, so digit-merging is structurally impossible.
- **6m**: accidental — nothing prevents digit merging, but TinyStories (children's stories) contains so few multi-digit numbers that BPE's frequency-driven merge selection never found a digit-pair common enough to merge. Feed it a corpus with lots of numbers and this behavior would likely disappear.

Same observable outcome, opposite causes: one is architecture, one is a side-effect of what the training data happened to contain. That distinction — "does this behavior hold *because it's designed to* or merely *because the data happened to make it likely*" — is worth checking every time a tokenizer or model appears to "just work" a certain way.

## Exercise 3 — Byte-level fallback: does it actually always work?

All three tokenizers here use a `ByteLevel` pre-tokenizer, and it's tempting to assume that alone guarantees graceful fallback on any text, since every byte can in principle be represented. **It doesn't, always** — run the cell below on a short Hindi (Devanagari) sentence and look closely at the output for all three.

Before you scroll to the mechanism note after the code: one of the three will produce mostly `<unk>` tokens and **fail to round-trip** (decode back to different, corrupted text). The other two will round-trip perfectly. Which one do you think fails, and why might "uses a byte-level pre-tokenizer" not be sufficient on its own?

In [ ]:
hindi = "नमस्ते, आप कैसे हैं?"
encode_all(hindi)

# Check round-tripping explicitly for each — does decode(encode(text)) == text?
print("6m round-trips:", tok_6m.decode(tok_6m.encode(hindi).ids) == hindi)
print("350m round-trips:", tok_350m.decode(tok_350m.encode(hindi).ids) == hindi)
print("50m round-trips:", tok_50m.decode(tok_50m.encode(hindi)) == hindi)

### The mechanism (verified, not assumed — check this against what you just ran)

`custom-gpt-6m` fails; `custom-gpt-350m` and `custom-gpt-50m` round-trip exactly. All three use a `ByteLevel` pre-tokenizer, so the difference isn't there — it's in how each vocabulary was **built**:

- `custom-gpt-350m/src/gpt/tokenizer.py` explicitly passes `initial_alphabet=pre_tokenizers.ByteLevel.alphabet()` to its `BpeTrainer` — this forces all 256 possible byte-values into the vocabulary from the start, whether or not they appeared in the training corpus. GPT-2's original tokenizer (what `custom-gpt-50m` uses via `tiktoken`) was built the same way.
- `custom-gpt-6m/src/gpt/data/prepare.py`'s `BpeTrainer` call has **no `initial_alphabet`** — its byte-level vocabulary only ends up containing the byte-values that actually appeared somewhere in its 100k-story English-only TinyStories training set. Devanagari's UTF-8 bytes never appeared, so they were never assigned single-byte tokens, and the tokenizer falls back to its configured `<unk>` (id 0) instead — which is lossy, unlike a genuine byte fallback.

**The lesson**: "byte-level pre-tokenizer" describes how text gets *split before* BPE runs, not what ends up *in* the final vocabulary — those are two different design decisions, and only forcing the full byte alphabet into the trainer guarantees the second one.

## Exercise 4 — Implement one round of BPE merging yourself, from scratch

Everything above used a trained library tokenizer as a black box. This exercise opens the box: implement **one merge step** of BPE by hand, so "the most frequent adjacent pair gets merged" stops being something you've read and becomes something you've made happen.

The skeleton below gives you a tiny toy corpus, already split into characters, and a working pair-counting function (that part's mechanical, not the point). **You write the merge step**: find the single most frequent adjacent pair across the whole corpus, and replace every occurrence of it with a new merged symbol.

Run your merge step 3-4 times in a row on the same corpus (feed its own output back in). What starts forming? Compare what your toy vocabulary looks like after a few rounds to real merges you saw in Exercise 1's output on real English — same underlying process, wildly different scale.

In [ ]:
from collections import Counter

# Toy corpus, pre-split into a list of "words", each word a list of symbols (starts as characters)
corpus = [list(w) for w in "the rabbit ran to the little forest the rabbit liked the forest".split()]
print("start:", corpus)

def count_pairs(corpus):
    """Count every adjacent symbol pair across the whole corpus. Working code — not the exercise."""
    pairs = Counter()
    for word in corpus:
        for a, b in zip(word, word[1:]):
            pairs[(a, b)] += 1
    return pairs

def merge_most_frequent(corpus):
    """TODO — this is the actual exercise. Implement:
    1. Find the single most frequent pair via count_pairs(corpus)
    2. Build a new corpus where every occurrence of that pair (as adjacent symbols
       within a word) is replaced by one merged symbol (e.g. ("t","h") -> "th")
    3. Return (new_corpus, the_pair_you_merged)
    """
    raise NotImplementedError("write this")

# Once merge_most_frequent works, run this a few times and watch the corpus change:
# corpus, merged = merge_most_frequent(corpus)
# print("merged:", merged, "->", corpus)

## Wrap-up

Write up at least one finding in `../docs/` using `TEMPLATE.md`'s shape — particularly Exercise 3's result if it surprised you (it's a real, verified bug-shaped gap in `custom-gpt-6m`'s tokenizer, not a hypothetical). Worth deciding: is this something worth fixing in that project (retraining with `initial_alphabet` forced), or is it a non-issue given TinyStories is English-only by design and the model will never see Hindi input anyway?

Where to go next: `../../docs/llm-engineering/09_tokenization.md` for the full mechanism writeup, or `../../docs/llm-engineering/23_the_serving_engine_ecosystem_vllm_and_friends.md` if you want to see how a tokenizer's format (or lack of a public preset) affects HF/vLLM conversion downstream.